PySpark DataFrame Case Study: Employee Performance Review Analysis
Scenario: You have a dataset containing employee performance reviews over time and want to analyze performance trends, average ratings, and identify high performers. The dataset contains the following columns:
• emp_id: Employee ID
• review_date: Date of the performance review
• department: Department name
• rating: Performance rating (1-5 scale)
• reviewer: Name of the person conducting the review
Sample Data

+--------+-------------+-------------+-------+------------+
| emp_id | review_date | department  | rating|  reviewer  |
+--------+-------------+-------------+-------+------------+
|   1    | 2024-01-10  | Engineering |   5   |   John     |
|   2    | 2024-01-11  | HR          |   4   |   Jane     |
|   3    | 2024-01-12  | Sales       |   3   |   Sam      |
|   4    | 2024-02-01  | Engineering |   5   |   John     |
|   1    | 2024-03-10  | Engineering |   4   |   Jane     |
|   2    | 2024-03-11  | HR          |  NULL |   Sam      |
+--------+-------------+-------------+-------+------------+


In [0]:
from pyspark.sql.functions import col, avg, coalesce, to_date, round
from pyspark.sql.window import Window


In [0]:
data = [
    (1, '2024-01-10', 'Engineering', 5, 'John'),
    (2, '2024-01-11', 'HR', 4, 'Jane'),
    (3, '2024-01-12', 'Sales', 3, 'Sam'),
    (4, '2024-02-01', 'Engineering', 5, 'John'),
    (1, '2024-03-10', 'Engineering', 4, 'Jane'),
    (2, '2024-03-11', 'HR', None, 'Sam')
]

columns = ["emp_id", "review_date", "department", "rating", "reviewer"] #Creating DataFrame

df = spark.createDataFrame(data, columns)

df = df.withColumn("review_date", to_date(col("review_date")))

display(df)

emp_id,review_date,department,rating,reviewer
1,2024-01-10,Engineering,5,John
2,2024-01-11,HR,4,Jane
3,2024-01-12,Sales,3,Sam
4,2024-02-01,Engineering,5,John
1,2024-03-10,Engineering,4,Jane
2,2024-03-11,HR,null,Sam


In [0]:
#1. Filter
#Find all reviews where the rating was 4 or higher.
df_filtered = df.filter(col("rating") >= 4)

display(df_filtered)

emp_id,review_date,department,rating,reviewer
1,2024-01-10,Engineering,5,John
2,2024-01-11,HR,4,Jane
4,2024-02-01,Engineering,5,John
1,2024-03-10,Engineering,4,Jane


In [0]:
#2. Handle Null Values
#Fill null values in the 'rating' column with the department average rating.

department_window = Window.partitionBy("department")

df_filled = df.withColumn(
    "rating_filled",
    coalesce(
        col("rating").cast("double"),
        avg("rating").over(department_window)
    )
)

display(df_filled)

emp_id,review_date,department,rating,reviewer,rating_filled
1,2024-01-10,Engineering,5,John,5.0
4,2024-02-01,Engineering,5,John,5.0
1,2024-03-10,Engineering,4,Jane,4.0
2,2024-01-11,HR,4,Jane,4.0
2,2024-03-11,HR,null,Sam,4.0
3,2024-01-12,Sales,3,Sam,3.0


In [0]:
#3. Drop Duplicates
#Remove duplicate reviews by 'emp_id' and 'review_date'.

df_no_duplicates = df.dropDuplicates(["emp_id", "review_date"])

display(df_no_duplicates)

emp_id,review_date,department,rating,reviewer
1,2024-01-10,Engineering,5,John
1,2024-03-10,Engineering,4,Jane
2,2024-01-11,HR,4,Jane
2,2024-03-11,HR,null,Sam
3,2024-01-12,Sales,3,Sam
4,2024-02-01,Engineering,5,John


In [0]:
#4. Select Specific Columns
#Select 'emp_id', 'department', and 'rating' columns.

df_selected = df.select("emp_id", "department", "rating")

display(df_selected)


emp_id,department,rating
1,Engineering,5
2,HR,4
3,Sales,3
4,Engineering,5
1,Engineering,4
2,HR,null


In [0]:
#5. Grouping and Aggregating
#Calculate the average rating per department.
df_grouped = df.groupBy("department") \
    .agg(round(avg("rating"), 2).alias("avg_rating"))

display(df_grouped)


department,avg_rating
Engineering,4.67
HR,4.0
Sales,3.0


In [0]:
#6. Joining DataFrames
#Join with another DataFrame 'df_employees' containing 'emp_id' and 'employee_name'.

df_joined = df.join(df_employees, on="emp_id", how="inner")

display(df_joined)


emp_id,review_date,department,rating,reviewer,employee_name
1,2024-01-10,Engineering,5,John,Alice
2,2024-01-11,HR,4,Jane,Bob
3,2024-01-12,Sales,3,Sam,Charlie
4,2024-02-01,Engineering,5,John,David
1,2024-03-10,Engineering,4,Jane,Alice
2,2024-03-11,HR,null,Sam,Bob


In [0]:
#7. Union of DataFrames
#Union with another 'df_new_reviews' DataFrame containing additional reviews.

new_review_data = [
    (5, '2024-04-01', 'Engineering', 5, 'Jane'),
    (6, '2024-04-05', 'Sales', 4, 'Sam')
]

df_new_reviews = spark.createDataFrame(new_review_data, columns)

df_new_reviews = df_new_reviews.withColumn("review_date", to_date(col("review_date")))

display(df_new_reviews)


emp_id,review_date,department,rating,reviewer
5,2024-04-01,Engineering,5,Jane
6,2024-04-05,Sales,4,Sam


In [0]:
df_union = df.unionByName(df_new_reviews)

display(df_union)

emp_id,review_date,department,rating,reviewer
1,2024-01-10,Engineering,5,John
2,2024-01-11,HR,4,Jane
3,2024-01-12,Sales,3,Sam
4,2024-02-01,Engineering,5,John
1,2024-03-10,Engineering,4,Jane
2,2024-03-11,HR,null,Sam
5,2024-04-01,Engineering,5,Jane
6,2024-04-05,Sales,4,Sam


In [0]:
#8. Temporary View and SQL
#Create a temp view and find the average rating for each employee.

df.createOrReplaceTempView("performance_reviews")

sql_result = spark.sql("""
    SELECT 
        emp_id,
        ROUND(AVG(rating), 2) AS avg_rating
    FROM performance_reviews
    GROUP BY emp_id
    ORDER BY emp_id
""")

display(sql_result)

emp_id,avg_rating
1,4.5
2,4.0
3,3.0
4,5.0


In [0]:
#9. Window Functions
#Calculate the cumulative average rating for each employee over time.
employee_window = Window.partitionBy("emp_id") \
    .orderBy("review_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_with_cumulative_avg = df.withColumn(
    "cumulative_avg",
    round(avg("rating").over(employee_window), 2)
)

display(df_with_cumulative_avg)


emp_id,review_date,department,rating,reviewer,cumulative_avg
1,2024-01-10,Engineering,5,John,5.0
1,2024-03-10,Engineering,4,Jane,4.5
2,2024-01-11,HR,4,Jane,4.0
2,2024-03-11,HR,null,Sam,4.0
3,2024-01-12,Sales,3,Sam,3.0
4,2024-02-01,Engineering,5,John,5.0
